# 🎯 [C204_M01] 전술 무전 음성 통신 환경 분석 및 딥페이크/스푸핑 음성 데이터셋 규격 설계
### Google Colab 실습 환경 (지휘통제(C2) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C204/c204_m01_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 3) |
| **미션 ID** | C204_M01 |
| **미션 주제** | 전술 무전 음성 통신 환경 분석 및 딥페이크/스푸핑 음성 데이터셋 규격 설계 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 지휘관 명의의 위조 음성·영상과 무전 기만이 실제로 보고되면서, 생성형 AI 시대에 '들리는 목소리가 정말 그 지휘관인가'를 확인하는 일이 지휘망 보안의 새로운 과제가 되었다. 본 프로젝트에서는 실제 음성 대신 탐지 학습용 합성 신호(배음 신호에 진짜 음성의 자연스러운 흔들림과 문헌상 알려진 위조 흔적을 통계적으로 표시한 추상 신호)를 코드로 만들어, 데이터셋 규격 설계, 무전 채널 잡음 모델링과 MFCC·스펙트로그램 전처리, 규칙 기반 Baseline, 신경망 판별 엔진, 진위 신뢰도 점수, C2 보안 상황판, 통신보안관 2차 검증(HITL), 전자전 재밍 스트레스 시험, AAR 재보정, 최종 통합까지 위조 음성 탐지 파이프라인을 단계별로 구현한다. 음성을 생성하거나 변조하는 공격 기법은 다루지 않으며, 오직 수신 측의 탐지·검증 기능만 구현한다. 최종 목표는 잡음이 심한 무전 환경에서도 위조 명령을 놓치지 않고 오경보를 줄이며, 모든 긴급 명령의 진위 확정이 사람의 인증·청음 확인을 거치도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c204_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c204.py` | 전처리, Baseline/신경망 모델, 신뢰도 모듈, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c204_data_spec.md`, `docs/c204_hitl_guide.md`, `docs/c204_aar_report.md`, `docs/c204_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c204_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c204_stress_test.py` | 전자전 재밍/잡음 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c204_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


# ----------------------------------------------------------------------------
# [군 표준 산출물 디렉토리 편철 준비 및 가상 데이터셋 무결성 확인]
# ----------------------------------------------------------------------------
output_dirs = [
    'data/schema',
    'docs',
    'src/preprocess',
    'src/models',
    'src/verification',
    'src/ui',
    'src/hitl',
    'tests',
    'configs'
]

print("=== [1] 군 표준 산출물 보관용 폴더 생성 ===")
for d in output_dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 디렉토리 준비 완료: {d}/")

print("\n=== [2] 가상 데이터셋 무결성 점검 ===")
data_dir = 'uploads/C204'
if not os.path.exists(data_dir):
    data_dir = '/home/subokim/Workspace/milicontent2/uploads/C204'

dataset_count = 0
for i in range(1, 11):
    fname = os.path.join(data_dir, f'c204_m{i:02d}_dataset.json')
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"  ✅ 데이터셋 확인: c204_m{i:02d}_dataset.json ({d.get('description', '')[:32]}...)")
        dataset_count += 1
    else:
        print(f"  ⚠️ 데이터셋 확인 필요: {fname}")

print(f"\n총 {dataset_count}/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!\n")


---

## 🎯 미션 01. 전술 무전 음성 통신 환경 분석 및 딥페이크/스푸핑 음성 데이터셋 규격 설계
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전술 무전 음성 통신 환경과 생성형 AI 기반 음성 합성(TTS/VC) 스푸핑 위협을 분석하고 데이터셋 규격을 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **전술 무전 환경과 생성형 AI 음성 위조(TTS/VC)·녹음 재생 위협을 탐지 단서 관점에서 정리하고, 무전 음성 클립 데이터셋의 입력 규격과 검증기를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **진짜·TTS 의심·VC 의심·재생 의심 4개 라벨과 유형별 탐지 단서를 표로 정리하고 규격화하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/c204_input_schema.json 및 docs/c204_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘통제(C2) 실무 및 러시아-우크라이나 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""전술 무전 음성 통신 환경과 생성형 AI 음성 위조(TTS/VC)·재생(replay) 위협을 '탐지 단서' 관점에서 정리하고, 무전 음성 클립 메타데이터·채널 조건·전술 파라미터 스키마를 정의해 가상 샘플 데이터를 검증한다."""
import json
import re
import numpy as np
import pandas as pd

# 1) 위협 유형별 탐지 단서(탐지 측 관점 요약, 생성 방법은 다루지 않음)
threats = pd.DataFrame([
    ("bonafide", "실제 송신자 음성", "자연스러운 피치·진폭 흔들림, 등록 화자와 일치"),
    ("tts_art", "문자->음성 합성 의심", "흔들림 부족, 고주파 에너지 부족, 프레임 경계 흔적"),
    ("vc_art", "음성 변환 의심", "배음 위상 불연속(HNR 저하), 스펙트럼 평탄화, 화자 특징 근사"),
    ("replay", "녹음 재생 의심", "스피커 대역 제한(저역 부족), 실내 잔향, 화자 특징은 일치"),
], columns=["라벨", "의미", "탐지 단서"])
print("[위협 유형별 탐지 단서]")
print(threats.to_string(index=False))
print("-> 무전 채널(300~3400Hz 대역 제한, 잡음, 끊김)이 단서를 가리므로 채널 조건을 함께 기록해야 판정 신뢰도를 해석할 수 있다.")

SCHEMA = {
    "clip": {
        "clip_id": {"type": str, "pattern": r"^C\d{6}$"},
        "net_id": {"type": str, "pattern": r"^NET-[A-Z]$"},                     # 가상 망 이름
        "sample_rate": {"type": int, "enum": [8000, 16000]},
        "duration_s": {"type": (int, float), "min": 0.2, "max": 60},
        "claimed_speaker": {"type": str, "pattern": r"^SPK_[A-Z]$"},
        "urgency": {"type": str, "enum": ["ROUTINE", "URGENT"]},
        "label": {"type": str, "enum": ["bonafide", "tts_art", "vc_art", "replay", "unknown"]},
    },
    "channel": {
        "band": {"type": str, "enum": ["HF", "VHF", "UHF"]},
        "codec": {"type": str, "enum": ["analog_fm", "digital_vocoder"]},
        "snr_db": {"type": (int, float), "min": -20, "max": 60},
        "dropout_ratio": {"type": (int, float), "min": 0, "max": 1},
    },
    "tactical": {
        "weather": {"type": str, "enum": ["clear", "rain", "snow", "fog", "night"]},
        "position_error_m": {"type": (int, float), "min": 0, "max": 500},
        "comm_delay_ms": {"type": (int, float), "min": 0, "max": 5000},
        "ew_jamming": {"type": bool},
    },
}


def check(name, value, rule):
    # [TODO : 학생 실습 1] 전술 무전 음성 통신 환경 분석 및 딥페이크/스푸핑 음성 데이터셋 규격 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def validate(rec):
    # [TODO : 학생 실습 1] 전술 무전 음성 통신 환경 분석 및 딥페이크/스푸핑 음성 데이터셋 규격 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rng = np.random.default_rng(1)
records = []
for i in range(20):
    records.append({"clip": {"clip_id": f"C01{i:04d}", "net_id": str(rng.choice(["NET-A", "NET-B"])), "sample_rate": 8000, "duration_s": 0.5,
                             "claimed_speaker": str(rng.choice(["SPK_A", "SPK_B", "SPK_C"])), "urgency": str(rng.choice(["ROUTINE", "URGENT"], p=[0.7, 0.3])),
                             "label": str(rng.choice(["bonafide", "tts_art", "vc_art", "replay"], p=[0.55, 0.15, 0.15, 0.15]))},
                    "channel": {"band": str(rng.choice(["VHF", "UHF"])), "codec": str(rng.choice(["analog_fm", "digital_vocoder"])),
                                "snr_db": float(round(rng.uniform(0, 25), 1)), "dropout_ratio": float(round(rng.uniform(0, 0.1), 2))},
                    "tactical": {"weather": str(rng.choice(["clear", "rain", "night"])), "position_error_m": float(round(rng.uniform(5, 50), 1)),
                                 "comm_delay_ms": int(rng.integers(50, 800)), "ew_jamming": bool(rng.random() < 0.2)}})
ok = sum(1 for r in records if not validate(r))
print(f"\n정상 샘플 {len(records)}건 중 검증 통과 {ok}건, 오류 {len(records) - ok}건")

bad = json.loads(json.dumps(records[0]))
bad["clip"]["clip_id"] = "CLIP-7"
bad["clip"]["sample_rate"] = 44100
bad["channel"]["snr_db"] = 99
del bad["tactical"]["comm_delay_ms"]
bad["tactical"]["ew_jamming"] = "yes"
print("[오류 주입 결과]")
for e in validate(bad):
    print(" -", e)
rule = json.loads(json.dumps(records[0]))
rule["clip"].update(urgency="URGENT", label="unknown"); rule["channel"]["snr_db"] = -3.0
print(" -", validate(rule)[0])

rows = []
for g in SCHEMA:
    for k, r in SCHEMA[g].items():
        t = r["type"].__name__ if isinstance(r["type"], type) else "number"
        rows.append((g, k, t, str({x: y for x, y in r.items() if x != "type"})[:52]))
print("\n[데이터 명세 요약]")
print(pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙"]).to_string(index=False))
print("\n라벨 분포:", pd.Series([r["clip"]["label"] for r in records]).value_counts().to_dict())


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[위협 유형별 탐지 단서]
      라벨           의미                                 탐지 단서
bonafide    실제 송신자 음성            자연스러운 피치·진폭 흔들림, 등록 화자와 일치
 tts_art 문자->음성 합성 의심         흔들림 부족, 고주파 에너지 부족, 프레임 경계 흔적
  vc_art     음성 변환 의심 배음 위상 불연속(HNR 저하), 스펙트럼 평탄화, 화자 특징 근사
  replay     녹음 재생 의심    스피커 대역 제한(저역 부족), 실내 잔향, 화자 특징은 일치
-> 무전 채널(300~3400Hz 대역 제한, 잡음, 끊김)이 단서를 가리므로 채널 조건을 함께 기록해야 판정 신뢰도를 해석할 수 있다.

정상 샘플 20건 중 검증 통과 20건, 오류 0건
[오류 주입 결과]
 - clip.clip_id: 형식 불일치 'CLIP-7'
 - clip.sample_rate: 허용되지 않은 값 '44100'
 - channel.snr_db: 최대값 60 초과
 - tactical.comm_delay_ms: 필수 필드 누락
 - tactical.ew_jamming: 자료형 오류(str)
 - 업무 규칙: SNR 0dB 미만 긴급 클립은 '판정 불가'로 표시 후 수동 확인 필요

[데이터 명세 요약]
      그룹               필드    자료형                                                   규칙
    clip          clip_id    str                             {'pattern': '^C\\d{6}$'}
    clip           net_id    str                           {'pattern': '^NET-[A-Z]$'}
    clip      sample_rate    int                              {'enum': [8000, 16000]}
    clip       duration_s number                              {'min': 0.2, 'max': 60}
    clip  claimed_speaker    str                           {'pattern': '^SPK_[A-Z]$'}
    clip          urgency    str                      {'enum': ['ROUTINE', 'URGENT']}
    clip            label    str {'enum': ['bonafide', 'tts_art', 'vc_art', 'replay',
 channel             band    str                       {'enum': ['HF', 'VHF', 'UHF']}
 channel            codec    str           {'enum': ['analog_fm', 'digital_vocoder']}
 channel           snr_db number                              {'min': -20, 'max': 60}
 channel    dropout_ratio number                                 {'min': 0, 'max': 1}
tactical          weather    str  {'enum': ['clear', 'rain', 'snow', 'fog', 'night']}
tactical position_error_m number                               {'min': 0, 'max': 500}
tactical    comm_delay_ms number                              {'min': 0, 'max': 5000}
tactical       ew_jamming   bool                                                   {}

라벨 분포: {'bonafide': 13, 'vc_art': 3, 'replay': 3, 'tts_art': 1}
```
</details>
